In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
hemdanovich_finall_path = kagglehub.dataset_download('hemdanovich/finall')

print('Data source import complete.')


In [ ]:
import os
from glob import glob
import pandas as pd
from PIL import Image
import torch
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer, TrainerCallback
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

# Define your labels and mappings
labels_list = ['angry', 'normal', 'tongue', 'happy']
label2id = {label: i for i, label in enumerate(labels_list)}
id2label = {i: label for i, label in enumerate(labels_list)}

# Initialize the image processor
processor = ViTImageProcessor.from_pretrained("dima806/facial_emotions_image_detection")


In [ ]:

base_dirs = {
    'angry': ['/kaggle/input/finall/finall/angry_brightness_decrease', '/kaggle/input/finall/finall/angry_brightness_increase'],
    'normal': ['/kaggle/input/finall/finall/normal_brightness_decrease', '/kaggle/input/finall/finall/normal_brightness_increase'],
    'tongue': ['/kaggle/input/finall/finall/tongue_brightness_decrease', '/kaggle/input/finall/finall/tongue_brightness_increase'],
    'happy': ['/kaggle/input/finall/finall/happy_brightness_decrease', '/kaggle/input/finall/finall/happy_brightness_increase']
}

file_paths = []
labels = []

for label, dirs in base_dirs.items():
    for dir_path in dirs:
        for file_path in glob(os.path.join(dir_path, "*.jpg")):
            file_paths.append(file_path)
            labels.append(label)

df = pd.DataFrame({"image_path": file_paths, "label": labels})
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'])


In [ ]:
def preprocess_and_save_images(df, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    processed_info = []

    for index, (idx, row) in enumerate(df.iterrows()):
        path, label = row['image_path'], row['label']
        image = Image.open(path).convert("RGB")
        processed_image = processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        tensor_path = os.path.join(save_dir, f"image_{idx}.pt")
        torch.save(processed_image, tensor_path)

        processed_info.append({"image_path": tensor_path, "labels": label2id[label]})

        print(f"Processed and saved image {index + 1}")

    return pd.DataFrame(processed_info)


In [ ]:
class DiskImageDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_tensor = torch.load(row['image_path'])
        label = row['labels']
        return {"pixel_values": image_tensor, "labels": label}


In [ ]:
# Specify directories for saving processed images
train_save_dir = '/content/processed_train_images'
test_save_dir = '/content/processed_test_images'

# Process and save the images to disk
processed_train_df = preprocess_and_save_images(train_df, train_save_dir)
processed_test_df = preprocess_and_save_images(test_df, test_save_dir)

# Create dataset instances
train_dataset = DiskImageDataset(processed_train_df)
test_dataset = DiskImageDataset(processed_test_df)


Processed and saved image 1
Processed and saved image 2
Processed and saved image 3
Processed and saved image 4
Processed and saved image 5
Processed and saved image 6
Processed and saved image 7
Processed and saved image 8
Processed and saved image 9
Processed and saved image 10
Processed and saved image 11
Processed and saved image 12
Processed and saved image 13
Processed and saved image 14
Processed and saved image 15
Processed and saved image 16
Processed and saved image 17
Processed and saved image 18
Processed and saved image 19
Processed and saved image 20
Processed and saved image 21
Processed and saved image 22
Processed and saved image 23
Processed and saved image 24
Processed and saved image 25
Processed and saved image 26
Processed and saved image 27
Processed and saved image 28
Processed and saved image 29
Processed and saved image 30
Processed and saved image 31
Processed and saved image 32
Processed and saved image 33
Processed and saved image 34
Processed and saved ima

In [ ]:
class CustomSaveModelCallback(TrainerCallback):
    def __init__(self, output_dir):
        super().__init__()
        self.output_dir = output_dir

    def on_evaluate(self, args, state, control, model=None, metrics=None, **kwargs):
        if metrics is not None and model is not None:
            epoch = state.epoch
            accuracy = metrics.get('eval_accuracy', 0.0)
            model_name = f"model_epoch_{epoch:.2f}_acc_{accuracy:.4f}.bin"
            model_path = os.path.join(self.output_dir, model_name)
            os.makedirs(self.output_dir, exist_ok=True)
            torch.save(model.state_dict(), model_path)
            print(f"Model saved to {model_path}")


In [ ]:
from torch.utils.data import DataLoader

# Define DataLoaders for training and testing datasets
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# You can now loop over train_loader and test_loader in your training and validation loops


In [ ]:
from transformers import ViTForImageClassification, TrainingArguments, Trainer
import os

# Assuming labels_list, id2label, and label2id are defined as before
labels_list = ['angry', 'normal', 'tongue', 'happy']
label2id = {label: i for i, label in enumerate(labels_list)}
id2label = {i: label for i, label in enumerate(labels_list)}

# Correctly initialize the model with custom labels and ignore mismatched sizes
model = ViTForImageClassification.from_pretrained(
    "dima806/facial_emotions_image_detection",
    num_labels=len(labels_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    learning_rate=0.00000008,  # Adjust the learning rate here
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)


# Define a function to compute metrics for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(-1)
    return {"accuracy": (predictions == labels).astype(float).mean()}

# Assuming train_dataset and test_dataset are defined using a custom dataset class compatible with Hugging Face's Trainer
# Example of such a dataset class is provided in the initial code setup

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[CustomSaveModelCallback(training_args.output_dir)]  # Assuming CustomSaveModelCallback is defined as before
)

# Start the training process

# You may optionally evaluate the model after training
# eval_results = trainer.evaluate()
# print(f"Evaluation Results: {eval_results}")


Some weights of ViTForImageClassification were not initialized from the model checkpoint at dima806/facial_emotions_image_detection and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([7]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([7, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_s

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(-1)
    return {"accuracy": (predictions == labels).astype(float).mean()}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[CustomSaveModelCallback(training_args.output_dir)]
)


In [ ]:
trainer.train()
trainer.evaluate()


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy
1,1.017900,0.958092,0.942692
2,0.745000,0.712311,0.990339
3,0.581900,0.563049,0.996267
4,0.482200,0.474865,0.998243
5,0.431800,0.428248,0.998829
6,0.415700,0.415190,0.998756


Model saved to ./results/model_epoch_1.00_acc_0.9427.bin


Checkpoint destination directory ./results/checkpoint-3416 already exists and is non-empty. Saving will proceed but saved results may be invalid.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Model saved to ./results/model_epoch_2.00_acc_0.9903.bin


...

/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Model saved to ./results/model_epoch_6.00_acc_0.9988.bin


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Model saved to ./results/model_epoch_6.00_acc_0.9988.bin


{'eval_loss': 0.42824795842170715,
 'eval_accuracy': 0.9988289541096391,
 'eval_runtime': 110.6268,
 'eval_samples_per_second': 123.505,
 'eval_steps_per_second': 7.72,
 'epoch': 6.0}